# Projeto #4: Marketing Mix Modeling (SOTA 2026)

## Projeto #4: Marketing Mix Modeling (MMM) — SOTA 2026 ⭐

**Domínio:** Marketing budgeting (CPG, Retail, SaaS)
**Pergunta:** Como aloco $1M de budget entre TV, Digital, Social e Outdoor?
**Conceitos cobertos:** TODOS os anteriores + Week 13 completa — Speculative
Decoding, Constitutional AI, Mixture of Experts, Synthetic Data, Fine-tuning
(LoRA). Este projeto é a integração final do curso.

Adaptado de `docs/source-material/07-projeto-mmm-raw.md` (código original tinha uma
referência a variável fora de escopo — corrigido aqui adicionando `elasticity`
como campo do estado).

In [ ]:
!pip install -q langgraph pydantic

import random
import asyncio
from typing import Optional
from pydantic import BaseModel, field_validator

### 1. Schema com Constitutional AI embutido (Week 13.2)

Validators do Pydantic funcionam como "constituição": o agente **não consegue**
gerar um mix que viole as regras de negócio.

In [ ]:
class MarketingMix(BaseModel):
    tv: float
    digital: float
    social: float
    outdoor: float

    @field_validator("tv", "digital", "social", "outdoor")
    @classmethod
    def min_spend(cls, v):
        if v < 10_000:
            raise ValueError("Mínimo $10k por canal (regra constitucional)")
        return v

    def total(self) -> float:
        return self.tv + self.digital + self.social + self.outdoor

class MMMState(BaseModel):
    budget: float = 1_000_000
    historical_data: list[dict] = []
    elasticity: dict = {}
    scenarios: list[dict] = []
    routing_strategy: Optional[str] = None
    best_mix: Optional[MarketingMix] = None
    expected_sales: float = 0
    confidence: float = 0

### 2. Dados sintéticos (Week 13.5 — Synthetic Data)

In [ ]:
def generate_historical_weeks(n: int = 20, seed: int = 11) -> list[dict]:
    random.seed(seed)
    weeks = []
    for w in range(n):
        tv, digital, social, outdoor = (
            random.uniform(30_000, 100_000),
            random.uniform(50_000, 150_000),
            random.uniform(10_000, 60_000),
            random.uniform(10_000, 40_000),
        )
        # diminishing returns simplificado
        sales = tv * 3.2 + digital * 4.1 + social * 2.0 + outdoor * 1.8
        sales *= random.uniform(0.9, 1.1)
        weeks.append({"week": w, "tv_spend": round(tv), "digital_spend": round(digital),
                       "social_spend": round(social), "outdoor_spend": round(outdoor),
                       "total_sales": round(sales)})
    return weeks

def generate_synthetic_scenarios(real_weeks: list[dict], factor: int = 5) -> list[dict]:
    """Expande o dataset real ~Nx com cenários sintéticos plausíveis."""
    synthetic = []
    for _ in range(len(real_weeks) * factor):
        base = random.choice(real_weeks)
        synthetic.append({k: v * random.uniform(0.85, 1.15) if "spend" in k or "sales" in k else v
                           for k, v in base.items()})
    return synthetic

historical = generate_historical_weeks()
print(f"✓ {len(historical)} semanas históricas + {len(generate_synthetic_scenarios(historical))} sintéticas")

### 3. Agent #1: Historical Analysis — calcula elasticidade

In [ ]:
async def historical_analysis(state: MMMState) -> MMMState:
    """MOCK — em produção, roda regressão real (ou pede pro Claude interpretar)."""
    state.elasticity = {"tv": 0.8, "digital": 1.1, "social": 0.6, "outdoor": 0.7}
    return state

### 4. Agent #2: Scenario Generation com Speculative Decoding (Week 13.1)

In [ ]:
async def draft_model_generate(budget: float, elasticity: dict, n: int = 200) -> list[dict]:
    """Draft rápido (equivalente a Claude Haiku): gera muitos candidatos baratos."""
    await asyncio.sleep(0.01)
    scenarios = []
    for _ in range(n):
        weights = {k: random.uniform(0.1, 1.0) * elasticity[k] for k in elasticity}
        total_w = sum(weights.values())
        mix = {k: round(budget * (w / total_w), 2) for k, w in weights.items()}
        scenarios.append(mix)
    return scenarios

async def verifier_model_verify(scenarios: list[dict], elasticity: dict, top_k: int = 20) -> list[dict]:
    """Verify preciso (equivalente a Claude Sonnet): só nos top-K candidatos."""
    await asyncio.sleep(0.01)
    for s in scenarios:
        s["predicted_sales"] = sum(s[ch] * elasticity[ch] for ch in elasticity)
        s["roi"] = s["predicted_sales"] / sum(s.values())
    return sorted(scenarios, key=lambda s: s["roi"], reverse=True)[:top_k]

async def scenario_generation(state: MMMState) -> MMMState:
    draft = await draft_model_generate(state.budget, state.elasticity, n=1000)
    verified = await verifier_model_verify(draft, state.elasticity, top_k=20)
    state.scenarios = verified
    return state

### 5. Agent #3: MoE Routing (Week 13.3) — escolhe estratégia por condição de mercado

In [ ]:
def moe_routing(state: MMMState, market_condition: str = "stable") -> MMMState:
    router = {"boom": "aggressive", "recession": "conservative"}
    state.routing_strategy = router.get(market_condition, "balanced")
    return state

### 6. Agent #4: Optimization — respeita a constituição (Week 13.2)

In [ ]:
def optimization(state: MMMState) -> MMMState:
    """Tenta os cenários em ordem de ROI até achar um que passe na validação
    Pydantic (constituição). Isso É o Constitutional AI na prática."""
    for scenario in state.scenarios:
        try:
            mix = MarketingMix(tv=scenario["tv"], digital=scenario["digital"],
                                social=scenario["social"], outdoor=scenario["outdoor"])
        except ValueError:
            continue  # viola a constituição, tenta o próximo
        state.best_mix = mix
        state.expected_sales = scenario["predicted_sales"]
        # roi aqui é "venda prevista por $ investido" (tipicamente 0.6-1.2 neste mock)
        state.confidence = round(min(0.95, max(0.3, scenario["roi"])), 2)
        break
    return state

### 7. Grafo completo (Week 6-7 aplicado ao projeto SOTA)

In [ ]:
from langgraph.graph import StateGraph, START, END

def node_analyze(state: MMMState) -> MMMState:
    return asyncio.run(historical_analysis(state))

def node_generate(state: MMMState) -> MMMState:
    return asyncio.run(scenario_generation(state))

def node_route(state: MMMState) -> MMMState:
    return moe_routing(state, market_condition="stable")

def node_optimize(state: MMMState) -> MMMState:
    return optimization(state)

graph = StateGraph(MMMState)
graph.add_node("analyze", node_analyze)
graph.add_node("generate", node_generate)
graph.add_node("route", node_route)
graph.add_node("optimize", node_optimize)
graph.add_edge(START, "analyze")
graph.add_edge("analyze", "generate")
graph.add_edge("generate", "route")
graph.add_edge("route", "optimize")
graph.add_edge("optimize", END)

mmm_agent = graph.compile()

### 8. Rodando

In [ ]:
initial = MMMState(budget=1_000_000, historical_data=historical)
result = mmm_agent.invoke(initial)
mix = result["best_mix"] if isinstance(result, dict) else result.best_mix
sales = result["expected_sales"] if isinstance(result, dict) else result.expected_sales
conf = result["confidence"] if isinstance(result, dict) else result.confidence
strategy = result["routing_strategy"] if isinstance(result, dict) else result.routing_strategy

print("🎯 Recommended Mix:")
print(f"   Strategy: {strategy}")
print(f"   TV:       ${mix.tv:,.0f}")
print(f"   Digital:  ${mix.digital:,.0f}")
print(f"   Social:   ${mix.social:,.0f}")
print(f"   Outdoor:  ${mix.outdoor:,.0f}")
print(f"   Total:    ${mix.total():,.0f}")
print(f"   Expected Sales: ${sales:,.0f}")
print(f"   Confidence: {conf:.0%}")

**Onde entram os 5 conceitos SOTA (Week 13):**

| Conceito | Onde no código |
|---|---|
| Speculative Decoding | `draft_model_generate` (1000 rápido) + `verifier_model_verify` (top 20 preciso) |
| Constitutional AI | `MarketingMix` validators + retry loop em `optimization` |
| Mixture of Experts | `moe_routing` — aggressive/balanced/conservative |
| Synthetic Data | `generate_synthetic_scenarios` — expande histórico 5x |
| Efficient Fine-tuning (LoRA) | Não implementado no mock (requer modelo local) — ver nota abaixo |

**Nota sobre LoRA:** fine-tuning não faz sentido mockado (não há modelo local
pra treinar). Em produção, isso seria feito adaptando um modelo aberto (não a
API da Anthropic) nos dados históricos da empresa — ver `docs/source-material/06-week13-advanced-raw.md`.

**Próximos passos pra produção:**
- Elasticidade real via regressão (statsmodels) ou o próprio Claude interpretando os dados
- Dataset real do BigQuery em vez do gerador sintético
- Deploy: Cloud Run + Cloud Scheduler pra rodar semanalmente